Step 1: Setup + GPU check

In [ ]:

!pip -q install shap sentencepiece

import torch, sys, platform
print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('PyTorch     :', torch.__version__)
print('CUDA avail. :', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then run again.'
print('GPU         :', torch.cuda.get_device_name(0))
print('GPU memory  :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

import importlib
for name in ['transformers','tokenizers','accelerate','datasets','sklearn','numpy','pandas','scipy','statsmodels','shap','sentencepiece']:
    try:
        m = importlib.import_module(name)
        print(f'{name:13s}:', getattr(m, '__version__', '?'))
    except Exception as e:
        print(f'{name:13s}: NOT INSTALLED ({e})')

# STEP 2: Configuration (everything that defines the experiment is here)


In [ ]:

import os, re, json, random, hashlib, time, glob, zipfile, warnings, shutil, gc
import numpy as np, pandas as pd, torch
warnings.filterwarnings('ignore')

# ---- experiment settings -------------------------------------------------
SEEDS    = [42, 43, 44]     # 3 seeds -> mean +- std (checklist #4). Use [42] for a quick test.
MAX_LEN  = 64
EPOCHS   = 3
LR       = 2e-5
WD       = 0.01
BATCH_TR = 32
BATCH_EV = 64
N_BOOT   = 2000             # bootstrap resamples
ALPHA    = 0.05

# ---- local output folder (NO Google Drive) -------------------------------
OUT = '/content/results'
os.makedirs(OUT, exist_ok=True)

# ---- model checkpoints ---------------------------------------------------
MODELS = {
    'mbert':          dict(hf='bert-base-multilingual-cased', weighted=True,  label='mBERT'),
    'xlmr':           dict(hf='xlm-roberta-base',             weighted=True,  label='XLM-RoBERTa'),
    'banglabert':     dict(hf='csebuetnlp/banglabert',        weighted=True,  label='BanglaBERT'),
    'banglabert_unw': dict(hf='csebuetnlp/banglabert',        weighted=False, label='BanglaBERT (unweighted)'),
    'indicbertv2':    dict(hf='ai4bharat/IndicBERTv2-MLM-only', weighted=True, label='IndicBERTv2'),
}

# ---- dataset integrity (SHA-256 first 16 hex, rows) ----------------------
EXPECTED = {'train': ('116c9e9eb7432dff', 40224),
            'val':   ('67995e80777818a1', 5028),
            'test':  ('4543a005555ee466', 5029)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---- hyperparameter table (for Appendix A) -------------------------------
HP = pd.DataFrame([
    ['TF-IDF word + SVM', 'word 1-2 gram, max 50,000 feat., sublinear TF; LinearSVC C=1.0 (probability-calibrated, 3-fold)'],
    ['TF-IDF char + SVM', 'char_wb 2-5 gram, max 100,000 feat., sublinear TF; LinearSVC C=1.0 (probability-calibrated, 3-fold)'],
    ['BiLSTM', 'emb 128, BiLSTM 128, dropout 0.4, Adam lr 1e-3, batch 64, up to 8 epochs, best val macro-F1, max_len 64'],
    ['Transformers', f'lr {LR}, weight decay {WD}, batch {BATCH_TR}, {EPOCHS} epochs, max_len {MAX_LEN}, fp16, best epoch by val macro-F1'],
    ['Loss', 'weighted cross-entropy (inverse class frequency); unweighted ablation for BanglaBERT'],
    ['Seeds', str(SEEDS)],
    ['Bootstrap', f'{N_BOOT} resamples, 95% percentile CI'],
    ['McNemar', 'continuity-corrected, Holm adjustment over all pairs'],
], columns=['Component', 'Setting'])
pd.set_option('display.max_colwidth', 200)
display(HP)
HP.to_csv(f'{OUT}/hyperparameters.csv', index=False)

# ---- run info (reproducibility log) --------------------------------------
import transformers, sklearn, scipy, statsmodels, platform
RUN_INFO = dict(
    date=time.strftime('%Y-%m-%d %H:%M:%S'), seeds=SEEDS, max_len=MAX_LEN, epochs=EPOCHS, lr=LR, wd=WD,
    batch_train=BATCH_TR, n_boot=N_BOOT, checkpoints={k: v['hf'] for k, v in MODELS.items()},
    python=platform.python_version(), torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
    transformers=transformers.__version__, sklearn=sklearn.__version__, numpy=np.__version__,
    pandas=pd.__version__, scipy=scipy.__version__, statsmodels=statsmodels.__version__,
    expected_data=EXPECTED)
json.dump(RUN_INFO, open(f'{OUT}/run_info.json', 'w'), indent=2)
print('\nSaved run_info.json and hyperparameters.csv ->', OUT)
print('Seeds:', SEEDS, '| max_len:', MAX_LEN, '| epochs:', EPOCHS)

# ---- helper: backup zip of everything so far (use after each big step) ---
def backup(name='backup'):
    zp = f'/content/{name}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(f'{OUT}/*'): z.write(f, os.path.basename(f))
    from google.colab import files
    files.download(zp)

# STEP 3: Load BD-SHS (no Drive) + integrity check + binary label check


In [ ]:

import urllib.request
from google.colab import files

DATA = '/content/bdshs'; os.makedirs(DATA, exist_ok=True)
BASE_URL = ''   # optional: public folder URL that contains train.csv / val.csv / test.csv

def have_all(): return all(os.path.exists(f'{DATA}/{k}.csv') for k in EXPECTED)

if not have_all() and BASE_URL:
    for k in EXPECTED:
        urllib.request.urlretrieve(f'{BASE_URL.rstrip("/")}/{k}.csv', f'{DATA}/{k}.csv')

if not have_all():
    print('Upload train.csv, val.csv, test.csv  (or the zip containing them)')
    up = files.upload()
    for name, content in up.items():
        p = f'/content/_up_{name}'; open(p, 'wb').write(content)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(p) as z: z.extractall('/content/_up_zip')
    for k in EXPECTED:
        hits = glob.glob(f'/content/_up_{k}.csv') + glob.glob(f'/content/_up_zip/**/{k}.csv', recursive=True)
        if hits: shutil.copy(hits[0], f'{DATA}/{k}.csv')
assert have_all(), 'train/val/test CSV not found - upload again'

# ---- 1. integrity: SHA-256 + row count ------------------------------------
def sha16(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16]
all_ok = True
for k, (h, n) in EXPECTED.items():
    raw = pd.read_csv(f'{DATA}/{k}.csv')
    ok = (sha16(f'{DATA}/{k}.csv') == h) and (len(raw) == n)
    all_ok &= ok
    print(f'{k:5s} rows={len(raw):6d}  sha256[:16]={sha16(f"{DATA}/{k}.csv")}  ->', 'OK' if ok else '*** MISMATCH ***')
print('\nDATASET INTEGRITY:', 'ALL OK' if all_ok else 'MISMATCH - do not use for thesis')

# ---- 2. columns and BINARY label check ------------------------------------
raw_tr = pd.read_csv(f'{DATA}/train.csv')
print('\nColumns:', list(raw_tr.columns))
for k in EXPECTED:
    d = pd.read_csv(f'{DATA}/{k}.csv')
    print(f'{k:5s} label values in "hate speech":', sorted(d['hate speech'].dropna().unique().tolist()),
          '| missing text:', int(d['sentence'].isna().sum()), '| missing label:', int(d['hate speech'].isna().sum()))

# ---- 3. clean text (same rule as before) ----------------------------------
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\u0980-\u09FF\s\w.,!?।]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load(k):
    d = pd.read_csv(f'{DATA}/{k}.csv').rename(columns={'sentence': 'text_raw', 'hate speech': 'label'})
    d['text'] = d['text_raw'].map(clean_text)       # only text + binary label are used for modelling
    d['label'] = d['label'].astype(int)
    return d[['text_raw', 'text', 'label']].reset_index(drop=True)

train, val, test = load('train'), load('val'), load('test')
y_test = test.label.values

# ---- 4. split summary table (Table 3.x in thesis) -------------------------
rows = []
for nm, d in [('Train', train), ('Validation', val), ('Test', test)]:
    rows.append(dict(Split=nm, Total=len(d), NonHate=int((d.label == 0).sum()), Hate=int((d.label == 1).sum()),
                     Hate_pct=round(100 * d.label.mean(), 2), Empty_after_cleaning=int((d.text.str.len() == 0).sum())))
split_tbl = pd.DataFrame(rows)
tot = split_tbl[['Total', 'NonHate', 'Hate']].sum()
split_tbl.loc[len(split_tbl)] = dict(Split='Total', Total=tot.Total, NonHate=tot.NonHate, Hate=tot.Hate,
                                     Hate_pct=round(100 * tot.Hate / tot.Total, 2),
                                     Empty_after_cleaning=split_tbl.Empty_after_cleaning.sum())
display(split_tbl)
split_tbl.to_csv(f'{OUT}/split_summary.csv', index=False)
np.save(f'{OUT}/y_test.npy', y_test)
train.to_pickle('/content/train.pkl'); val.to_pickle('/content/val.pkl'); test.to_pickle('/content/test.pkl')
print('\nSaved split_summary.csv. Example cleaned rows:')
display(train.sample(3, random_state=0)[['text', 'label']])

# STEP 3b: Data audit - duplicates, train/test overlap, label conflicts


In [ ]:

def norm(s): return s.str.strip()

# ---- 0. the empty-after-cleaning rows ---------------------------------------
emp = train[train.text.str.len() == 0]
print('Rows empty after cleaning (train):', len(emp))
display(emp[['text_raw', 'label']])

# work only on non-empty texts
tr, va, te = [d[d.text.str.len() > 0].copy() for d in (train, val, test)]
for d in (tr, va, te): d['t'] = norm(d.text)

# ---- 1. duplicates INSIDE each split ----------------------------------------
def within(d):
    g = d.groupby('t').label.agg(['count', 'nunique'])
    extra = int((g['count'] - 1).clip(lower=0).sum())          # surplus copies
    conflict_groups = int(((g['count'] > 1) & (g['nunique'] > 1)).sum())
    return extra, conflict_groups

# ---- 2. overlap BETWEEN splits (later split rows whose text is in earlier) ---
def overlap(a, b, name):
    m = b.merge(a[['t', 'label']].drop_duplicates(), on='t', how='inner', suffixes=('', '_other'))
    rows_in_b = b.t.isin(set(a.t))
    n = int(rows_in_b.sum())
    # label disagreement: row's label differs from at least one label of that text in a
    lab_sets = a.groupby('t').label.agg(set)
    diff = int(sum(1 for t, l in zip(b.t[rows_in_b], b.label[rows_in_b]) if l not in lab_sets[t] or len(lab_sets[t]) > 1))
    return dict(Pair=name, Rows_checked=len(b), Rows_with_text_in_other=n,
                Pct=round(100 * n / len(b), 2), Of_which_label_conflict=diff)

rows = []
for nm, d in [('Train', tr), ('Validation', va), ('Test', te)]:
    ex, cg = within(d)
    rows.append(dict(Check=f'{nm}: duplicate copies inside split', Count=ex, Extra=f'{cg} text(s) carry conflicting labels'))
display(pd.DataFrame(rows))

ov = pd.DataFrame([overlap(tr, va, 'Validation rows also in Train'),
                   overlap(tr, te, 'Test rows also in Train'),
                   overlap(va, te, 'Test rows also in Validation')])
display(ov)

# ---- 3. save masks / tables -------------------------------------------------
train_texts = set(tr.t)
test_in_train = np.array([(len(t.strip()) > 0) and (t.strip() in train_texts) for t in test.text])
np.save(f'{OUT}/test_in_train_mask.npy', test_in_train)
pd.DataFrame(rows).to_csv(f'{OUT}/audit_within_split.csv', index=False)
ov.to_csv(f'{OUT}/audit_between_splits.csv', index=False)
print('\nTest rows whose text also occurs in Train:', int(test_in_train.sum()), 'of', len(test))

# STEP 3c: Truncation rate at max_len = 64 for each tokenizer


In [ ]:

from transformers import AutoTokenizer

all_df = pd.concat([train.assign(split='train'), val.assign(split='val'), test.assign(split='test')], ignore_index=True)
texts = all_df.text.tolist()

rows = []
for key, cfg in MODELS.items():
    if key == 'banglabert_unw':      # same tokenizer as banglabert
        continue
    try:
        tk = AutoTokenizer.from_pretrained(cfg['hf'])
        lens = np.array([len(x) for x in tk(texts, add_special_tokens=True, truncation=False)['input_ids']])
        rows.append(dict(Tokenizer=cfg['label'], Checkpoint=cfg['hf'], Mean_len=round(lens.mean(), 1),
                         Median=int(np.median(lens)), P95=int(np.percentile(lens, 95)), Max=int(lens.max()),
                         Truncated_n=int((lens > MAX_LEN).sum()), Truncated_pct=round(100 * (lens > MAX_LEN).mean(), 2)))
    except Exception as e:
        rows.append(dict(Tokenizer=cfg['label'], Checkpoint=cfg['hf'], Mean_len=None, Median=None, P95=None,
                         Max=None, Truncated_n=None, Truncated_pct=f'ERROR: {repr(e)[:80]}'))

# whitespace words (used by the BiLSTM)
wl = np.array([len(t.split()) for t in texts])
rows.append(dict(Tokenizer='Whitespace words (BiLSTM)', Checkpoint='-', Mean_len=round(wl.mean(), 1), Median=int(np.median(wl)),
                 P95=int(np.percentile(wl, 95)), Max=int(wl.max()), Truncated_n=int((wl > MAX_LEN).sum()),
                 Truncated_pct=round(100 * (wl > MAX_LEN).mean(), 2)))
trunc = pd.DataFrame(rows)
display(trunc)
trunc.to_csv(f'{OUT}/truncation_rate.csv', index=False)
print('Comments analysed:', len(texts), '| max_len =', MAX_LEN)

# STEP 2-patch: change max_len to 128 and re-check truncation


In [ ]:

MAX_LEN = 128
HP.loc[HP.Component == 'BiLSTM', 'Setting'] = HP.loc[HP.Component == 'BiLSTM', 'Setting'].str.replace('max_len 64', 'max_len 128')
HP.loc[HP.Component == 'Transformers', 'Setting'] = f'lr {LR}, weight decay {WD}, batch {BATCH_TR}, {EPOCHS} epochs, max_len {MAX_LEN}, fp16, best epoch by val macro-F1'
HP.to_csv(f'{OUT}/hyperparameters.csv', index=False)
RUN_INFO['max_len'] = MAX_LEN
json.dump(RUN_INFO, open(f'{OUT}/run_info.json', 'w'), indent=2)

from transformers import AutoTokenizer
rows = []
for key, cfg in MODELS.items():
    if key == 'banglabert_unw': continue
    tk = AutoTokenizer.from_pretrained(cfg['hf'])
    lens = np.array([len(x) for x in tk(all_df.text.tolist(), add_special_tokens=True, truncation=False)['input_ids']])
    rows.append(dict(Tokenizer=cfg['label'], Truncated_n=int((lens > MAX_LEN).sum()), Truncated_pct=round(100 * (lens > MAX_LEN).mean(), 2)))
wl = np.array([len(t.split()) for t in all_df.text])
rows.append(dict(Tokenizer='Whitespace words (BiLSTM)', Truncated_n=int((wl > MAX_LEN).sum()), Truncated_pct=round(100 * (wl > MAX_LEN).mean(), 2)))
t128 = pd.DataFrame(rows); display(t128); t128.to_csv(f'{OUT}/truncation_rate_128.csv', index=False)
print('MAX_LEN is now', MAX_LEN)

# STEP 4: Metrics + helpers to save / load predictions (probabilities)


In [ ]:

assert MAX_LEN == 128, 'MAX_LEN is not 128 yet -> run the Step 2-patch cell first (or the RESUME script).'

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             precision_score, recall_score)

def metrics(y, p):
    """All numbers that go into the thesis tables, from true labels y and predicted labels p."""
    mp, mr, mf, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    return dict(
        Accuracy    = accuracy_score(y, p),
        Macro_P     = mp,  Macro_R = mr,  Macro_F1 = mf,                 # main metric
        Weighted_F1 = f1_score(y, p, average='weighted'),               # comparable with Romim et al.
        Hate_P      = precision_score(y, p, pos_label=1, zero_division=0),
        Hate_R      = recall_score(y, p, pos_label=1, zero_division=0),
        Hate_F1     = f1_score(y, p, pos_label=1, zero_division=0))

def save_probs(name, seed, probs):  np.save(f'{OUT}/probs_{name}_s{seed}.npy', probs)
def have(name, seed):               return os.path.exists(f'{OUT}/probs_{name}_s{seed}.npy')
def get(name, seed):                return np.load(f'{OUT}/probs_{name}_s{seed}.npy')

RUN_LOG = f'{OUT}/results_per_run.csv'
def log_run(name, seed, probs):
    """Compute metrics on the test set for one (model, seed) and append to results_per_run.csv."""
    m = metrics(y_test, probs.argmax(1)); m.update(Model=name, Seed=seed)
    row = pd.DataFrame([m])[['Model', 'Seed'] + [c for c in m if c not in ('Model', 'Seed')]]
    row.to_csv(RUN_LOG, mode='a', header=not os.path.exists(RUN_LOG), index=False)
    print(name, 'seed', seed, {k: round(v, 4) for k, v in m.items() if k not in ('Model', 'Seed')})

# ---- sanity checks (so we trust the function before using it) --------------
# 1) perfect prediction must give all 1.0
print('perfect   :', {k: round(v, 4) for k, v in metrics(y_test, y_test).items()})
# 2) predicting all "non-hate" gives the majority-class baseline
print('all-0     :', {k: round(v, 4) for k, v in metrics(y_test, np.zeros_like(y_test)).items()})
# 3) random guessing should be near 0.5
rng = np.random.RandomState(0)
print('random    :', {k: round(v, 4) for k, v in metrics(y_test, rng.randint(0, 2, len(y_test))).items()})
print('\nStep 4 ready. Results of every run will be appended to', RUN_LOG)

# STEP 5: TF-IDF + SVM baselines (word, char) + reference run with sklearn default tokenizer


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

C_GRID = [0.1, 0.3, 1, 3]
VECS = {
  'tfidf_word_svm':       ('TF-IDF word (1-2) + SVM',
                           lambda: TfidfVectorizer(token_pattern=r'\S+', ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
  'tfidf_char_svm':       ('TF-IDF char (2-5) + SVM',
                           lambda: TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=100000, sublinear_tf=True)),
  'tfidf_word_defaulttok':('[reference] word SVM, sklearn default tokenizer',
                           lambda: TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
}
ytr, yva = train.label.values, val.label.values
best_C, tune_rows = {}, []

for name, (label, mk) in VECS.items():
    if all(have(name, s) for s in SEEDS):
        print('skip (already done):', name); continue
    vec = mk()
    Xtr = vec.fit_transform(train.text); Xva = vec.transform(val.text); Xte = vec.transform(test.text)
    # --- tune C on VALIDATION only
    scores = {}
    for C in C_GRID:
        clf = LinearSVC(C=C, random_state=42).fit(Xtr, ytr)
        scores[C] = f1_score(yva, clf.predict(Xva), average='macro')
        tune_rows.append(dict(Model=name, C=C, Val_Macro_F1=round(scores[C], 4)))
    best_C[name] = max(scores, key=scores.get)
    print(f'{name}: features={Xtr.shape[1]}  val macro-F1 by C = {({c: round(v, 4) for c, v in scores.items()})}  -> best C = {best_C[name]}')
    # --- final fit with probability calibration, once per seed
    for s in SEEDS:
        if have(name, s): continue
        set_seed(s)
        clf = CalibratedClassifierCV(LinearSVC(C=best_C[name], random_state=s), cv=3).fit(Xtr, ytr)
        pr = clf.predict_proba(Xte); save_probs(name, s, pr); log_run(name, s, pr)

if tune_rows:
    pd.DataFrame(tune_rows).to_csv(f'{OUT}/svm_C_tuning_validation.csv', index=False)
    json.dump(best_C, open(f'{OUT}/svm_best_C.json', 'w'))
    HP.loc[HP.Component == 'TF-IDF word + SVM', 'Setting'] = f"word 1-2 gram (whitespace tokens), max 50,000 feat., sublinear TF; LinearSVC C={best_C.get('tfidf_word_svm')} chosen on validation (grid {C_GRID}), probability-calibrated 3-fold"
    HP.loc[HP.Component == 'TF-IDF char + SVM', 'Setting'] = f"char_wb 2-5 gram, max 100,000 feat., sublinear TF; LinearSVC C={best_C.get('tfidf_char_svm')} chosen on validation (grid {C_GRID}), probability-calibrated 3-fold"
    HP.to_csv(f'{OUT}/hyperparameters.csv', index=False)
print('\nStep 5 done. Results so far:'); display(pd.read_csv(RUN_LOG).round(4))

# STEP 6: BiLSTM (random-init embeddings), 3 seeds, best epoch chosen on validation


In [ ]:

import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
dev = torch.device('cuda')

# class weights from the TRAIN set (same weights as the transformers use later)
cw = len(train) / (2 * np.bincount(train.label.values))
CW = torch.tensor(cw, dtype=torch.float)
print('class weights (non-hate, hate):', cw.round(4))

VOCAB_MAX, MIN_FREQ, BL_EPOCHS, BL_BATCH, BL_LR = 50000, 2, 8, 64, 1e-3
cnt = Counter(w for t in train.text for w in t.split())
vocab = {w: i + 2 for i, (w, c) in enumerate(cnt.most_common(VOCAB_MAX)) if c >= MIN_FREQ}
print('BiLSTM vocabulary size:', len(vocab))

def enc(t):
    ids = [vocab.get(w, 1) for w in t.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))
def tens(d): return torch.tensor([enc(t) for t in d.text]), torch.tensor(d.label.values)
Xtr, Ytr = tens(train); Xva, Yva = tens(val); Xte, _ = tens(test)

class BiLSTM(nn.Module):
    def __init__(s, V):
        super().__init__()
        s.emb = nn.Embedding(V + 2, 128, padding_idx=0)
        s.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        s.drop = nn.Dropout(0.4); s.fc = nn.Linear(256, 2)
    def forward(s, x):
        o, _ = s.lstm(s.emb(x)); m = (x != 0).unsqueeze(-1).float()
        return s.fc(s.drop((o * m).sum(1) / m.sum(1).clamp(min=1)))   # mean over real (non-pad) tokens

def predict_logits(model, X):
    model.eval(); out = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(X), batch_size=256): out.append(model(xb.to(dev)).cpu())
    return torch.cat(out)

epoch_log = []
for s in SEEDS:
    if have('bilstm', s): print('skip (already done): bilstm seed', s); continue
    set_seed(s)
    g = torch.Generator(); g.manual_seed(s)
    model = BiLSTM(len(vocab)).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=BL_LR)
    lossf = nn.CrossEntropyLoss(weight=CW.to(dev))
    best_f1, best_state, best_ep = -1, None, 0
    for ep in range(1, BL_EPOCHS + 1):
        model.train(); tot = 0
        for xb, yb in DataLoader(TensorDataset(Xtr, Ytr), batch_size=BL_BATCH, shuffle=True, generator=g):
            opt.zero_grad(); loss = lossf(model(xb.to(dev)), yb.to(dev)); loss.backward(); opt.step(); tot += loss.item() * len(xb)
        vf = f1_score(Yva.numpy(), predict_logits(model, Xva).argmax(1).numpy(), average='macro')
        epoch_log.append(dict(Seed=s, Epoch=ep, Train_loss=round(tot / len(Xtr), 4), Val_Macro_F1=round(vf, 4)))
        if vf > best_f1: best_f1, best_ep, best_state = vf, ep, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pr = torch.softmax(predict_logits(model, Xte), 1).numpy()
    save_probs('bilstm', s, pr); log_run('bilstm', s, pr)
    print(f'   seed {s}: best epoch = {best_ep} (val macro-F1 {best_f1:.4f})')

if epoch_log:
    el = pd.DataFrame(epoch_log); el.to_csv(f'{OUT}/bilstm_epoch_log.csv', index=False)
    display(el.pivot(index='Epoch', columns='Seed', values='Val_Macro_F1'))
    HP.loc[HP.Component == 'BiLSTM', 'Setting'] = f'random-init emb 128, BiLSTM 128 (mean pooling), dropout 0.4, Adam lr {BL_LR}, batch {BL_BATCH}, up to {BL_EPOCHS} epochs, best epoch by val macro-F1, weighted CE, max_len {MAX_LEN} words, vocab {len(vocab)}'
    HP.to_csv(f'{OUT}/hyperparameters.csv', index=False)
print('\nStep 6 done.'); display(pd.read_csv(RUN_LOG).query("Model == 'bilstm'").round(4))

In [ ]:
# SAVE EVERYTHING DONE SO FAR (results + all tables as text)
def show(title, path):
    s = f'\n===== {title} =====\n'
    s += pd.read_csv(path).round(4).to_string(index=False) if os.path.exists(path) else '(not found)'
    return s

report  = 'BD-SHS single-corpus run - progress report (Steps 1 to 6)\n'
report += f'Saved: {time.strftime("%Y-%m-%d %H:%M:%S")}\n'
report += '\n===== Settings (run_info.json) =====\n' + json.dumps(json.load(open(f'{OUT}/run_info.json')), indent=2)
report += show('Hyperparameters', f'{OUT}/hyperparameters.csv')
report += show('Dataset split summary', f'{OUT}/split_summary.csv')
report += show('Audit: duplicates inside each split', f'{OUT}/audit_within_split.csv')
report += show('Audit: overlap between splits', f'{OUT}/audit_between_splits.csv')
report += show('Truncation at max_len 64', f'{OUT}/truncation_rate.csv')
report += show('Truncation at max_len 128 (final)', f'{OUT}/truncation_rate_128.csv')
report += show('SVM: C tuning on validation', f'{OUT}/svm_C_tuning_validation.csv')
report += show('BiLSTM: validation F1 per epoch', f'{OUT}/bilstm_epoch_log.csv')
report += show('RESULTS per run (test set)', f'{OUT}/results_per_run.csv')

rr = pd.read_csv(f'{OUT}/results_per_run.csv')
mean = rr.groupby('Model')[['Accuracy', 'Macro_F1', 'Weighted_F1', 'Hate_P', 'Hate_R', 'Hate_F1']].agg(['mean', 'std']).round(4)
report += '\n\n===== RESULTS mean and std over seeds =====\n' + mean.to_string()

open(f'{OUT}/PROGRESS_REPORT.txt', 'w', encoding='utf-8').write(report)
print(report)
backup('progress_after_step6')

In [ ]:
# STATE CHECK
try:
    print('Colab session OK. train rows:', len(train), '| MAX_LEN:', MAX_LEN, '| seeds:', SEEDS)
    print('GPU:', torch.cuda.get_device_name(0))
    print('Finished runs:'); display(pd.read_csv(RUN_LOG).groupby('Model').size().rename('n_seeds_done'))
except NameError as e:
    print('SESSION WAS RESET ->', e, '\nUse the RESUME script (Blocks A, B, C, D).')

# HELPER (run once): save_progress(tag) = write report + zip everything + download


In [ ]:

def save_progress(tag):
    rep  = f'BD-SHS single-corpus run - progress "{tag}"  ({time.strftime("%Y-%m-%d %H:%M:%S")})\n'
    rep += '\n===== Settings =====\n' + json.dumps(json.load(open(f'{OUT}/run_info.json')), indent=2)
    if os.path.exists(RUN_LOG):
        rr = pd.read_csv(RUN_LOG)
        rep += '\n\n===== Results per run (test set) =====\n' + rr.round(4).to_string(index=False)
        rep += '\n\n===== Mean and std over seeds =====\n' + \
               rr.groupby('Model')[['Accuracy', 'Macro_F1', 'Weighted_F1', 'Hate_P', 'Hate_R', 'Hate_F1']].agg(['mean', 'std']).round(4).to_string()
    for f in sorted(glob.glob(f'{OUT}/epochlog_*.csv')):
        rep += f'\n\n===== {os.path.basename(f)} =====\n' + pd.read_csv(f).to_string(index=False)
    open(f'{OUT}/PROGRESS_REPORT.txt', 'w', encoding='utf-8').write(rep)
    n = len(os.listdir(OUT))
    print(f'[saved] {n} files in {OUT}  -> downloading progress_{tag}.zip')
    backup(f'progress_{tag}')
print('save_progress ready')

# STEP 7a: Fine-tune ONE transformer to test the pipeline (mBERT, seed 42)


In [ ]:

import torch.nn as nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)
from datasets import Dataset

assert MAX_LEN == 128
CW = torch.tensor(len(train) / (2 * np.bincount(train.label.values)), dtype=torch.float)

class WeightedTrainer(Trainer):
    use_w = True
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels'); out = model(**inputs)
        w = CW.to(out.logits.device) if self.use_w else None
        loss = nn.CrossEntropyLoss(weight=w)(out.logits.float(), labels)
        return (loss, out) if return_outputs else loss

def run_transformer(name, seed):
    cfg = MODELS[name]; hf = cfg['hf']
    set_seed(seed); t0 = time.time()
    tk = AutoTokenizer.from_pretrained(hf)
    def mk(d):
        ds = Dataset.from_dict({'text': d.text.tolist(), 'labels': d.label.tolist()})
        return ds.map(lambda b: tk(b['text'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['text'])
    dtr, dva, dte = mk(train), mk(val), mk(test)
    model = AutoModelForSequenceClassification.from_pretrained(hf, num_labels=2)
    ckpt = f'/content/ckpt/{name}_s{seed}'
    args = TrainingArguments(output_dir=ckpt, num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=WD,
        per_device_train_batch_size=BATCH_TR, per_device_eval_batch_size=BATCH_EV,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True,
        metric_for_best_model='f1', greater_is_better=True, seed=seed, data_seed=seed,
        fp16=True, report_to='none', disable_tqdm=True, logging_strategy='epoch')
    def cm(ep):
        p = ep.predictions[0] if isinstance(ep.predictions, tuple) else ep.predictions
        return {'f1': f1_score(ep.label_ids, p.argmax(-1), average='macro')}
    tr = WeightedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                         processing_class=tk, compute_metrics=cm)
    tr.use_w = cfg['weighted']
    tr.train()
    ep_log = [dict(Model=name, Seed=seed, Epoch=round(h['epoch']), Val_Macro_F1=round(h['eval_f1'], 4))
              for h in tr.state.log_history if 'eval_f1' in h]
    pd.DataFrame(ep_log).to_csv(f'{OUT}/epochlog_{name}_s{seed}.csv', index=False)
    def probs(ds):
        p = tr.predict(ds).predictions; p = p[0] if isinstance(p, tuple) else p
        return torch.softmax(torch.tensor(p).float(), 1).numpy()
    pr, pv = probs(dte), probs(dva)
    np.save(f'{OUT}/valprobs_{name}_s{seed}.npy', pv)
    save_probs(name, seed, pr); log_run(name, seed, pr)
    print(f'   best epoch by val F1 = {tr.state.best_model_checkpoint.split("-")[-1] if tr.state.best_model_checkpoint else "?"} | '
          f'time {(time.time() - t0) / 60:.1f} min')
    if name == 'banglabert' and seed == SEEDS[0]:          # keep one BanglaBERT for SHAP later
        tr.save_model('/content/models/banglabert'); tk.save_pretrained('/content/models/banglabert')
    del tr, model; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(ckpt, ignore_errors=True)

if not have('mbert', 42):
    run_transformer('mbert', 42)
print('\nStep 7a done.'); display(pd.read_csv(RUN_LOG).query("Model == 'mbert'").round(4))

# STEP 7-fix: keep the best epoch's weights IN MEMORY (no checkpoint reload)


In [ ]:

from transformers import TrainerCallback

class KeepBest(TrainerCallback):
    def __init__(self): self.best, self.state, self.epoch = -1, None, 0
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        f = (metrics or {}).get('eval_f1', -1)
        if f > self.best and model is not None:
            self.best, self.epoch = f, int(round(state.epoch))
            self.state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def run_transformer(name, seed):
    cfg = MODELS[name]; hf = cfg['hf']
    set_seed(seed); t0 = time.time()
    tk = AutoTokenizer.from_pretrained(hf)
    def mk(d):
        ds = Dataset.from_dict({'text': d.text.tolist(), 'labels': d.label.tolist()})
        return ds.map(lambda b: tk(b['text'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['text'])
    dtr, dva, dte = mk(train), mk(val), mk(test)
    model = AutoModelForSequenceClassification.from_pretrained(hf, num_labels=2)
    args = TrainingArguments(output_dir=f'/content/ckpt/{name}_s{seed}', num_train_epochs=EPOCHS, learning_rate=LR,
        weight_decay=WD, per_device_train_batch_size=BATCH_TR, per_device_eval_batch_size=BATCH_EV,
        eval_strategy='epoch', save_strategy='no', load_best_model_at_end=False, seed=seed, data_seed=seed,
        fp16=True, report_to='none', disable_tqdm=True, logging_strategy='epoch')
    def cm(ep):
        p = ep.predictions[0] if isinstance(ep.predictions, tuple) else ep.predictions
        return {'f1': f1_score(ep.label_ids, p.argmax(-1), average='macro')}
    kb = KeepBest()
    tr = WeightedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                         processing_class=tk, compute_metrics=cm, callbacks=[kb])
    tr.use_w = cfg['weighted']
    tr.train()
    tr.model.load_state_dict(kb.state)                       # <- restore the BEST epoch exactly
    ep_log = [dict(Model=name, Seed=seed, Epoch=int(round(h['epoch'])), Val_Macro_F1=round(h['eval_f1'], 4))
              for h in tr.state.log_history if 'eval_f1' in h]
    pd.DataFrame(ep_log).to_csv(f'{OUT}/epochlog_{name}_s{seed}.csv', index=False)
    def probs(ds):
        p = tr.predict(ds).predictions; p = p[0] if isinstance(p, tuple) else p
        return torch.softmax(torch.tensor(p).float(), 1).numpy()
    pr, pv = probs(dte), probs(dva)
    np.save(f'{OUT}/valprobs_{name}_s{seed}.npy', pv)
    # sanity: validation macro-F1 of the restored model must equal the best epoch's logged value
    vf = f1_score(val.label.values, pv.argmax(1), average='macro')
    print(f'   best epoch = {kb.epoch} | logged val F1 {kb.best:.4f} | re-computed val F1 {vf:.4f} | time {(time.time()-t0)/60:.1f} min')
    save_probs(name, seed, pr); log_run(name, seed, pr)
    if name == 'banglabert' and seed == SEEDS[0]:
        tr.save_model('/content/models/banglabert'); tk.save_pretrained('/content/models/banglabert')
    del tr, model, kb; gc.collect(); torch.cuda.empty_cache()
print('run_transformer updated')

In [ ]:
# ============================================================================
# BD-SHS single-corpus FINAL run  --  ONE-CELL RESUME  (paste in ONE Colab cell, run once)
# Runtime > Change runtime type > T4 GPU first.
# DRIVE_BACKUP = True  : personal convenience - results are copied to YOUR Drive after every save_progress(),
#                        and restored automatically here. Set False when sharing the notebook.
# ============================================================================
DRIVE_BACKUP = True
DOWNLOAD_ZIP = False      # True = also download a zip to your computer at every save

# ---- 1. install + GPU ----
!pip -q install shap sentencepiece
import torch, sys, platform, importlib
print('Python      :', sys.version.split()[0]); print('PyTorch     :', torch.__version__)
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then run again.'
print('GPU         :', torch.cuda.get_device_name(0))
for name in ['transformers','tokenizers','accelerate','datasets','sklearn','numpy','pandas','scipy','statsmodels','shap','sentencepiece']:
    try:
        m = importlib.import_module(name); print(f'{name:13s}:', getattr(m, '__version__', '?'))
    except Exception as e: print(f'{name:13s}: NOT INSTALLED ({e})')


# ---- 2. config ----
import os, re, json, random, hashlib, time, glob, zipfile, warnings, shutil, gc
import numpy as np, pandas as pd, torch
warnings.filterwarnings('ignore')

SEEDS    = [42, 43, 44]
MAX_LEN  = 128
EPOCHS   = 3
LR       = 2e-5
WD       = 0.01
BATCH_TR = 32
BATCH_EV = 64
N_BOOT   = 2000
ALPHA    = 0.05
OUT = '/content/results'; os.makedirs(OUT, exist_ok=True)

MODELS = {
    'mbert':          dict(hf='bert-base-multilingual-cased', weighted=True,  label='mBERT'),
    'xlmr':           dict(hf='xlm-roberta-base',             weighted=True,  label='XLM-RoBERTa'),
    'banglabert':     dict(hf='csebuetnlp/banglabert',        weighted=True,  label='BanglaBERT'),
    'banglabert_unw': dict(hf='csebuetnlp/banglabert',        weighted=False, label='BanglaBERT (unweighted)'),
    'indicbertv2':    dict(hf='ai4bharat/IndicBERTv2-MLM-only', weighted=True, label='IndicBERTv2'),
}
EXPECTED = {'train': ('116c9e9eb7432dff', 40224), 'val': ('67995e80777818a1', 5028), 'test': ('4543a005555ee466', 5029)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

HP = pd.DataFrame([
    ['TF-IDF word + SVM', 'word 1-2 gram, max 50,000 feat., sublinear TF; LinearSVC C=1.0 (probability-calibrated, 3-fold)'],
    ['TF-IDF char + SVM', 'char_wb 2-5 gram, max 100,000 feat., sublinear TF; LinearSVC C=1.0 (probability-calibrated, 3-fold)'],
    ['BiLSTM', f'emb 128, BiLSTM 128, dropout 0.4, Adam lr 1e-3, batch 64, up to 8 epochs, best val macro-F1, max_len {MAX_LEN}'],
    ['Transformers', f'lr {LR}, weight decay {WD}, batch {BATCH_TR}, {EPOCHS} epochs, max_len {MAX_LEN}, fp16, best epoch by val macro-F1'],
    ['Loss', 'weighted cross-entropy (inverse class frequency); unweighted ablation for BanglaBERT'],
    ['Seeds', str(SEEDS)],
    ['Bootstrap', f'{N_BOOT} resamples, 95% percentile CI'],
    ['McNemar', 'continuity-corrected, Holm adjustment over all pairs'],
], columns=['Component', 'Setting'])
pd.set_option('display.max_colwidth', 200); display(HP); HP.to_csv(f'{OUT}/hyperparameters.csv', index=False)

import transformers, sklearn, scipy, statsmodels, platform
RUN_INFO = dict(date=time.strftime('%Y-%m-%d %H:%M:%S'), seeds=SEEDS, max_len=MAX_LEN, epochs=EPOCHS, lr=LR, wd=WD,
    batch_train=BATCH_TR, n_boot=N_BOOT, checkpoints={k: v['hf'] for k, v in MODELS.items()},
    python=platform.python_version(), torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
    transformers=transformers.__version__, sklearn=sklearn.__version__, numpy=np.__version__,
    pandas=pd.__version__, scipy=scipy.__version__, statsmodels=statsmodels.__version__, expected_data=EXPECTED)
json.dump(RUN_INFO, open(f'{OUT}/run_info.json', 'w'), indent=2)


# ---- backup(): copy results to Drive (optional) and/or download a zip ----
DRIVE_DIR = '/content/drive/MyDrive/BDSHS_thesis_data'
def backup(name='backup'):
    if DRIVE_BACKUP:
        try:
            os.makedirs(f'{DRIVE_DIR}/results_backup', exist_ok=True)
            shutil.copytree(OUT, f'{DRIVE_DIR}/results_backup', dirs_exist_ok=True)
            print(f'[Drive] results copied to {DRIVE_DIR}/results_backup ({len(os.listdir(OUT))} files)')
        except Exception as e:
            print('[Drive] copy FAILED:', repr(e))
    if DOWNLOAD_ZIP or not DRIVE_BACKUP:
        zp = f'/content/{name}.zip'
        with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in glob.glob(f'{OUT}/*'): z.write(f, os.path.basename(f))
        from google.colab import files; files.download(zp)


# ---- 3. restore + data ----

# ---- restore previous results from Drive ----
from google.colab import files
if DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    bk = f'{DRIVE_DIR}/results_backup'
    if os.path.isdir(bk):
        shutil.copytree(bk, OUT, dirs_exist_ok=True); print('Restored', len(os.listdir(OUT)), 'result files from Drive')
    else:
        print('No Drive backup yet -> starting fresh (or upload a progress zip below)')
        print('Optional: upload a progress zip (Cancel to skip)')
        try:
            for n, c in files.upload().items():
                if n.lower().endswith('.zip') and 'archive' not in n.lower():
                    open(f'/content/_bk_{n}', 'wb').write(c)
                    with zipfile.ZipFile(f'/content/_bk_{n}') as z: z.extractall(OUT)
                    print('restored', n)
        except Exception as e: print('no zip restored:', repr(e))

# ---- BD-SHS data: Drive copy first, else upload ----
DATA = '/content/bdshs'; os.makedirs(DATA, exist_ok=True)
BASE_URL = ''
def have_all(): return all(os.path.exists(f'{DATA}/{k}.csv') for k in EXPECTED)
if DRIVE_BACKUP and not have_all():
    for k in EXPECTED:
        p = f'{DRIVE_DIR}/{k}.csv'
        if os.path.exists(p): shutil.copy(p, f'{DATA}/{k}.csv')
    if not have_all():
        for z in glob.glob(f'{DRIVE_DIR}/archive*.zip'):
            with zipfile.ZipFile(z) as zz: zz.extractall('/content/_up_zip')
            for k in EXPECTED:
                hits = glob.glob(f'/content/_up_zip/**/{k}.csv', recursive=True)
                if hits: shutil.copy(hits[0], f'{DATA}/{k}.csv')
if not have_all() and BASE_URL:
    import urllib.request
    for k in EXPECTED: urllib.request.urlretrieve(f'{BASE_URL.rstrip("/")}/{k}.csv', f'{DATA}/{k}.csv')
if not have_all():
    print('Upload archive__3_.zip  (or train.csv, val.csv, test.csv)')
    for name, content in files.upload().items():
        p = f'/content/_up_{name}'; open(p, 'wb').write(content)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(p) as z: z.extractall('/content/_up_zip')
    for k in EXPECTED:
        hits = glob.glob(f'/content/_up_{k}.csv') + glob.glob(f'/content/_up_zip/**/{k}.csv', recursive=True)
        if hits: shutil.copy(hits[0], f'{DATA}/{k}.csv')
    if DRIVE_BACKUP:                      # keep a copy so next time nothing needs uploading
        os.makedirs(DRIVE_DIR, exist_ok=True)
        for k in EXPECTED: shutil.copy(f'{DATA}/{k}.csv', f'{DRIVE_DIR}/{k}.csv')
assert have_all(), 'train/val/test CSV not found'

def sha16(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16]
all_ok = True
for k, (h, n) in EXPECTED.items():
    raw = pd.read_csv(f'{DATA}/{k}.csv'); ok = (sha16(f'{DATA}/{k}.csv') == h) and (len(raw) == n); all_ok &= ok
    print(f'{k:5s} rows={len(raw):6d}  sha256[:16]={sha16(f"{DATA}/{k}.csv")}  ->', 'OK' if ok else '*** MISMATCH ***')
print('DATASET INTEGRITY:', 'ALL OK' if all_ok else 'MISMATCH - do not use for thesis')

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\u0980-\u09FF\s\w.,!?।]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def load(k):
    d = pd.read_csv(f'{DATA}/{k}.csv').rename(columns={'sentence': 'text_raw', 'hate speech': 'label'})
    d['text'] = d['text_raw'].map(clean_text); d['label'] = d['label'].astype(int)
    return d[['text_raw', 'text', 'label']].reset_index(drop=True)
train, val, test = load('train'), load('val'), load('test')
y_test = test.label.values; np.save(f'{OUT}/y_test.npy', y_test)
all_df = pd.concat([train.assign(split='train'), val.assign(split='val'), test.assign(split='test')], ignore_index=True)
print({'train': len(train), 'val': len(val), 'test': len(test)})


# ---- 4. helpers ----
# Run this AFTER block C (and after restoring your backup zip there).
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, precision_score, recall_score)

def metrics(y, p):
    mp, mr, mf, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    return dict(Accuracy=accuracy_score(y, p), Macro_P=mp, Macro_R=mr, Macro_F1=mf,
                Weighted_F1=f1_score(y, p, average='weighted'),
                Hate_P=precision_score(y, p, pos_label=1, zero_division=0),
                Hate_R=recall_score(y, p, pos_label=1, zero_division=0),
                Hate_F1=f1_score(y, p, pos_label=1, zero_division=0))
def save_probs(name, seed, probs):  np.save(f'{OUT}/probs_{name}_s{seed}.npy', probs)
def have(name, seed):               return os.path.exists(f'{OUT}/probs_{name}_s{seed}.npy')
def get(name, seed):                return np.load(f'{OUT}/probs_{name}_s{seed}.npy')
RUN_LOG = f'{OUT}/results_per_run.csv'
def log_run(name, seed, probs):
    m = metrics(y_test, probs.argmax(1)); m.update(Model=name, Seed=seed)
    row = pd.DataFrame([m])[['Model', 'Seed'] + [c for c in m if c not in ('Model', 'Seed')]]
    row.to_csv(RUN_LOG, mode='a', header=not os.path.exists(RUN_LOG), index=False)
    print(name, 'seed', seed, {k: round(v, 4) for k, v in m.items() if k not in ('Model', 'Seed')})

# reload the settings that were updated during Steps 2-patch, 5 and 6 (SVM C, BiLSTM details, MAX_LEN=128)
if os.path.exists(f'{OUT}/hyperparameters.csv'): HP = pd.read_csv(f'{OUT}/hyperparameters.csv')
if os.path.exists(f'{OUT}/run_info.json'):       RUN_INFO = json.load(open(f'{OUT}/run_info.json'))
assert MAX_LEN == 128 and RUN_INFO.get('max_len') == 128, 'MAX_LEN should be 128'
print('Restored. Finished runs so far:')
if os.path.exists(RUN_LOG): display(pd.read_csv(RUN_LOG).groupby('Model').size().rename('n_seeds_done'))



# ---- 5. step 7 definitions ----
import torch.nn as nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, TrainerCallback)
from datasets import Dataset

assert MAX_LEN == 128
CW = torch.tensor(len(train) / (2 * np.bincount(train.label.values)), dtype=torch.float)

class WeightedTrainer(Trainer):
    use_w = True
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels'); out = model(**inputs)
        w = CW.to(out.logits.device) if self.use_w else None
        loss = nn.CrossEntropyLoss(weight=w)(out.logits.float(), labels)
        return (loss, out) if return_outputs else loss

class KeepBest(TrainerCallback):
    def __init__(self): self.best, self.state, self.epoch = -1, None, 0
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        f = (metrics or {}).get('eval_f1', -1)
        if f > self.best and model is not None:
            self.best, self.epoch = f, int(round(state.epoch))
            self.state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def run_transformer(name, seed):
    cfg = MODELS[name]; hf = cfg['hf']
    set_seed(seed); t0 = time.time()
    tk = AutoTokenizer.from_pretrained(hf)
    def mk(d):
        ds = Dataset.from_dict({'text': d.text.tolist(), 'labels': d.label.tolist()})
        return ds.map(lambda b: tk(b['text'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['text'])
    dtr, dva, dte = mk(train), mk(val), mk(test)
    model = AutoModelForSequenceClassification.from_pretrained(hf, num_labels=2)
    args = TrainingArguments(output_dir=f'/content/ckpt/{name}_s{seed}', num_train_epochs=EPOCHS, learning_rate=LR,
        weight_decay=WD, per_device_train_batch_size=BATCH_TR, per_device_eval_batch_size=BATCH_EV,
        eval_strategy='epoch', save_strategy='no', load_best_model_at_end=False, seed=seed, data_seed=seed,
        fp16=True, report_to='none', disable_tqdm=True, logging_strategy='epoch')
    def cm(ep):
        p = ep.predictions[0] if isinstance(ep.predictions, tuple) else ep.predictions
        return {'f1': f1_score(ep.label_ids, p.argmax(-1), average='macro')}
    kb = KeepBest()
    tr = WeightedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                         processing_class=tk, compute_metrics=cm, callbacks=[kb])
    tr.use_w = cfg['weighted']
    tr.train()
    tr.model.load_state_dict(kb.state)
    ep_log = [dict(Model=name, Seed=seed, Epoch=int(round(h['epoch'])), Val_Macro_F1=round(h['eval_f1'], 4))
              for h in tr.state.log_history if 'eval_f1' in h]
    pd.DataFrame(ep_log).to_csv(f'{OUT}/epochlog_{name}_s{seed}.csv', index=False)
    def probs(ds):
        p = tr.predict(ds).predictions; p = p[0] if isinstance(p, tuple) else p
        return torch.softmax(torch.tensor(p).float(), 1).numpy()
    pr, pv = probs(dte), probs(dva)
    np.save(f'{OUT}/valprobs_{name}_s{seed}.npy', pv)
    vf = f1_score(val.label.values, pv.argmax(1), average='macro')
    print(f'   best epoch = {kb.epoch} | logged val F1 {kb.best:.4f} | re-computed val F1 {vf:.4f} | time {(time.time()-t0)/60:.1f} min')
    save_probs(name, seed, pr); log_run(name, seed, pr)
    if name == 'banglabert' and seed == SEEDS[0]:
        tr.save_model('/content/models/banglabert'); tk.save_pretrained('/content/models/banglabert')
    del tr, model, kb; gc.collect(); torch.cuda.empty_cache()

def save_progress(tag):
    rep  = f'BD-SHS single-corpus run - progress "{tag}"  ({time.strftime("%Y-%m-%d %H:%M:%S")})\n'
    rep += '\n===== Settings =====\n' + json.dumps(json.load(open(f'{OUT}/run_info.json')), indent=2)
    if os.path.exists(RUN_LOG):
        rr = pd.read_csv(RUN_LOG)
        rep += '\n\n===== Results per run (test set) =====\n' + rr.round(4).to_string(index=False)
        rep += '\n\n===== Mean and std over seeds =====\n' + \
               rr.groupby('Model')[['Accuracy', 'Macro_F1', 'Weighted_F1', 'Hate_P', 'Hate_R', 'Hate_F1']].agg(['mean', 'std']).round(4).to_string()
    for f in sorted(glob.glob(f'{OUT}/epochlog_*.csv')):
        rep += f'\n\n===== {os.path.basename(f)} =====\n' + pd.read_csv(f).to_string(index=False)
    open(f'{OUT}/PROGRESS_REPORT.txt', 'w', encoding='utf-8').write(rep)
    print(f'[saved] {len(os.listdir(OUT))} files in {OUT}  -> downloading progress_{tag}.zip')
    backup(f'progress_{tag}')
print('Block E ready: run_transformer, save_progress, WeightedTrainer defined.')

print('\nALL READY. Next: run the next model cell (7b-x).')

In [ ]:
backup('after_restore')


# STEP 7b-1: mBERT, all seeds (finished ones are skipped)

In [ ]:

for s in SEEDS:
    if have('mbert', s): print('skip (already done): mbert seed', s); continue
    run_transformer('mbert', s)
rr = pd.read_csv(RUN_LOG)
print('\nmBERT so far:'); display(rr[rr.Model == 'mbert'].round(4))
print(rr[rr.Model == 'mbert'][['Accuracy', 'Macro_F1', 'Hate_P', 'Hate_R']].agg(['mean', 'std']).round(4))
save_progress('after_7b1_mbert')

# STEP 7b-2: XLM-RoBERTa, all seeds (finished ones are skipped)


In [ ]:

for s in SEEDS:
    if have('xlmr', s): print('skip (already done): xlmr seed', s); continue
    run_transformer('xlmr', s)
rr = pd.read_csv(RUN_LOG)
print('\nXLM-R so far:'); display(rr[rr.Model == 'xlmr'].round(4))
print(rr[rr.Model == 'xlmr'][['Accuracy', 'Macro_F1', 'Hate_P', 'Hate_R']].agg(['mean', 'std']).round(4))
save_progress('after_7b2_xlmr')

# STEP 7b-3: BanglaBERT (weighted loss), all seeds (finished ones are skipped)


In [ ]:

for s in SEEDS:
    if have('banglabert', s): print('skip (already done): banglabert seed', s); continue
    run_transformer('banglabert', s)
rr = pd.read_csv(RUN_LOG)
print('\nBanglaBERT so far:'); display(rr[rr.Model == 'banglabert'].round(4))
print(rr[rr.Model == 'banglabert'][['Accuracy', 'Macro_F1', 'Hate_P', 'Hate_R']].agg(['mean', 'std']).round(4))
print('\nSaved model for SHAP:', os.path.exists('/content/models/banglabert'))
save_progress('after_7b3_banglabert')